In [2]:
using DifferentialEquations
using Plots
using FFMPEG

# Defining Non-Dimensionalized(ND) Parameters
G = 6.67408e-20      # Univ. Gravitational Constant [km3 kg-1 s-2]
mEarth = 5.97219e24  # Mass of the Earth [kg]
mMoon = 7.34767e22   # Mass of the Moon [kg]
a = 3.844e5          # Semi-major axis of Earth and Moon [km]
m1 = mEarth
m2 = mMoon
M_ = m1 + m2      # ND Mass Parameter
L_ = a            # ND Length Parameter
T_ = (L_^3 / (G * M_))^(1 / 2)  # ND Time Parameter
μ = m2 / M_

# CR3BP Model
function model_CR3BP(du, state, p, t)
    μ = p
    x = state[1]
    y = state[2]
    z = state[3]
    ẋ = state[4]
    ẏ = state[5]
    ż = state[6]
    ẍ = 2*ẏ + x - (((1-μ)*(x+μ))/((x+μ)^2 + y^2 + z^2)^(3/2)) - ((μ*(x-1+μ))/((((x-1+μ)^2) + y^2 + z^2)^(3/2)))
    ÿ = -2*ẋ + y - (((1-μ)*y)/((x+μ)^2 + y^2 + z^2)^(3/2)) - ((μ*y)/((((x-1+μ)^2) + y^2 + z^2)^(3/2)))
    z̈ = ((-(1-μ)*z)/((x+μ)^2 + y^2 + z^2)^(3/2)) - (((μ*z)/(((x-1+μ)^2) + y^2 + z^2)^(3/2)))
    du .= [ẋ, ẏ, ż, ẍ, ÿ, z̈]
end

# Initial Conditions [Initial State Vector]
X_0 = 50000 / L_   # ND x
Y_0 = 0               # ND y
Z_0 = 0               # ND z
VX_0 = 1.08 * T_ / L_  # ND x_dot
VY_0 = 3.18 * T_ / L_  # ND y_dot
VZ_0 = 0.68 * T_ / L_  # ND z_dot
state_0 = [X_0, Y_0, Z_0, VX_0, VY_0, VZ_0]  # ND ICs

# Time Array
tspan = (0.0, 50.0)
t = range(tspan[1], tspan[2], length=1000)

# Numerically Integrating
prob = ODEProblem(model_CR3BP, state_0, tspan, μ)
sol = solve(prob, Tsit5(), saveat=t, reltol=1e-9, abstol=1e-9)

# Rotational Frame Position Time History
X_rot = sol[1, :]
Y_rot = sol[2, :]
Z_rot = sol[3, :]

# Inertial Frame Position Time History
X_Iner = sol[1, :] .* cos.(t) .- sol[2, :] .* sin.(t)
Y_Iner = sol[1, :] .* sin.(t) .+ sol[2, :] .* cos.(t)
Z_Iner = sol[3, :]

# Constant m1 and m2 Rotational Frame Locations for CR3BP Primaries
m1_loc = [-μ, 0, 0]
m2_loc = [1 - μ, 0, 0]

# Moving m1 and m2 Inertial Locations for CR3BP Primaries
X_m1 = m1_loc[1] .* cos.(t) .- m1_loc[2] .* sin.(t)
Y_m1 = m1_loc[1] .* sin.(t) .+ m1_loc[2] .* cos.(t)
Z_m1 = m1_loc[3] .* ones(length(t))
X_m2 = m2_loc[1] .* cos.(t) .- m2_loc[2] .* sin.(t)
Y_m2 = m2_loc[1] .* sin.(t) .+ m2_loc[2] .* cos.(t)
Z_m2 = m2_loc[3] .* ones(length(t))

# Create the animation
anim = @animate for i in 1:length(t)
    plt_rot = plot3d(X_rot[1:i], Y_rot[1:i], Z_rot[1:i], c=:red, label="Trajectory")
    scatter!(plt_rot, [m1_loc[1]], [m1_loc[2]], [m1_loc[3]], c=:blue, marker=:circle, label="Earth")
    scatter!(plt_rot, [m2_loc[1]], [m2_loc[2]], [m2_loc[3]], c=:grey, marker=:circle, label="Moon")
    title!(plt_rot, "Rotating Frame CR3BP Visualization (μ = $(round(μ, digits=6)))")
    xlabel!(plt_rot, "x [ND]")
    ylabel!(plt_rot, "y [ND]")
    zlabel!(plt_rot, "z [ND]")

    plt_iner = plot3d(X_Iner[1:i], Y_Iner[1:i], Z_Iner[1:i], c=:red, label="Trajectory")
    scatter!(plt_iner, [X_m1[i]], [Y_m1[i]], [Z_m1[i]], c=:blue, marker=:circle, label="Earth")
    scatter!(plt_iner, [X_m2[i]], [Y_m2[i]], [Z_m2[i]], c=:grey, marker=:circle, label="Moon")
    title!(plt_iner, "Inertial Frame CR3BP Visualization (μ = $(round(μ, digits=6)))")
    xlabel!(plt_iner, "X [ND]")
    ylabel!(plt_iner, "Y [ND]")
    zlabel!(plt_iner, "Z [ND]")

    plot(plt_rot, plt_iner, layout=(2, 1))
end

# Save the animation
mp4(anim, "trajectory_animation.mp4", fps = 15)


┌ Warning: Backwards compatability support of the new return codes to Symbols will be deprecated with the Julia v1.9 release. Please see https://docs.sciml.ai/SciMLBase/stable/interfaces/Solutions/#retcodes for more information
└ @ SciMLBase C:\Users\anura\.julia\packages\SciMLBase\QqtZA\src\retcodes.jl:355
┌ Info: Saved animation to 
└   fn = "C:\\Users\\anura\\trajectory_animation.mp4"


Plots.AnimatedGif("C:\\Users\\anura\\trajectory_animation.mp4")

In [3]:
println(sol)

ODESolution{Float64, 2, Vector{Vector{Float64}}, Nothing, Nothing, Vector{Float64}, Vector{Vector{Vector{Float64}}}, ODEProblem{Vector{Float64}, Tuple{Float64, Float64}, true, Float64, ODEFunction{true, SciMLBase.AutoSpecialize, typeof(model_CR3BP), LinearAlgebra.UniformScaling{Bool}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, typeof(SciMLBase.DEFAULT_OBSERVED), Nothing, Nothing}, Base.Pairs{Symbol, Union{}, Tuple{}, NamedTuple{(), Tuple{}}}, SciMLBase.StandardODEProblem}, Tsit5{typeof(OrdinaryDiffEq.trivial_limiter!), typeof(OrdinaryDiffEq.trivial_limiter!), Static.False}, OrdinaryDiffEq.InterpolationData{ODEFunction{true, SciMLBase.AutoSpecialize, typeof(model_CR3BP), LinearAlgebra.UniformScaling{Bool}, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, Nothing, typeof(SciMLBase.DEFAULT_OBSERVED), Nothing, Nothing}, Vector{Vector{Float64}}, Vector{Float64

0.10144151151776865, 0.8534142988025952, -0.8060170430834906, 0.20528743997931795], [0.23547450000388653, -0.48884486446361114, 0.11081297322872273, 0.7122887326811246, -0.753618292458049, 0.17041188363876952], [0.26791238732509315, -0.5255991879076534, 0.1186018160367093, 0.5861769337235385, -0.7169801103764315, 0.14168781744006978], [0.294354463061939, -0.5607542701039271, 0.1250694162277355, 0.4722920750999546, -0.6888066624631384, 0.11738738161809738], [0.3153575665157739, -0.5946179975880596, 0.13040717742604246, 0.3685374721559251, -0.6648336883695527, 0.09638760412799614], [0.3313882537082425, -0.6273303385577995, 0.13476003555937988, 0.27337737072800977, -0.6424592412460886, 0.07792520465625215], [0.3428472936466984, -0.658927324720473, 0.13824076516486772, 0.18569751814467855, -0.6200495857057382, 0.061461590836399314], [0.35008785136096227, -0.6893792059950932, 0.14093909570537352, 0.10469394950435411, -0.5965638338990266, 0.046604524690933666], [0.3534288483342244, -0.718614

 0.6463165227942137, 0.12955476213404973, 0.3028053131827204, 0.37916675697101593, -0.07184193359747053], [-0.7095177959850416, 0.6644377164029045, 0.12578561849514228, 0.3413332559929229, 0.3443014129262325, -0.07873909058907493], [-0.6915453698833898, 0.6807167866806193, 0.12167617802141012, 0.3762251285119806, 0.30557933724789604, -0.08544393535369378], [-0.6719210305333202, 0.6949642561993163, 0.1172355287815988, 0.40731643752893903, 0.26314185698459697, -0.09197745550700052], [-0.6508390639042051, 0.706997600225255, 0.11247177771311688, 0.434445258038044, 0.21712489301084476, -0.09835791357735987], [-0.6285017503251685, 0.7166408823491609, 0.10739218141285604, 0.45745441100923256, 0.16765507521937018, -0.1046010504381499], [-0.6051190749104249, 0.7237241732077183, 0.10200326977723374, 0.476193535609307, 0.11484492020251226, -0.1107201674185397], [-0.5809083338357826, 0.7280826984182226, 0.09631096880064101, 0.49052120832694507, 0.05878677395216126, -0.11672607409962851], [-0.55609

-0.1302335054043195, -0.22736986402323728], [0.8122410481523722, -0.47008644130049776, -0.11440975115771601, -0.4340397622346754, -0.08287271900678715, -0.22084215329090978], [0.7902021261227468, -0.4729964112454194, -0.12527996904356228, -0.44616900778853336, -0.032994242388689025, -0.2133661313322363], [0.767626169901256, -0.47334793668747815, -0.13575028310527942, -0.45550107772523973, 0.0193561149975145, -0.20484412735430912], [0.7446526793667424, -0.47101816769924304, -0.14576548038294607, -0.46206184973589554, 0.0741462970874166, -0.19515809387744915], [0.7214188970155136, -0.465885430199836, -0.15526384714297287, -0.4659171236029048, 0.13136207500055955, -0.18416553750341316], [0.6980575667807465, -0.4578282261385735, -0.16417581863399353, -0.46718280165778436, 0.19101138420684735, -0.1716942230341936], [0.6746941048624879, -0.44672401248558624, -0.17242232620943793, -0.46603840105770405, 0.2531287943401769, -0.15753530600615925], [0.651442982847169, -0.4324477589205164, -0.1799

[-0.7091893303924057, 0.8015988014323048, -0.02347291847568433, 0.4002829093692221, 0.21996710349064819, -0.24179172244962524], [-0.6887927717221839, 0.8117654629201492, -0.035546035264023744, 0.4142951771736404, 0.18599168155779475, -0.24056332368658057], [-0.6677660290917862, 0.8201875931236864, -0.04754450435614685, 0.4254493447095263, 0.15027834101785295, -0.23880568837059826], [-0.6462544372576394, 0.8267816144947607, -0.05944133093271433, 0.4336555155269431, 0.11296302796285515, -0.23649777875246866], [-0.6244076027752158, 0.831470721000943, -0.07120834342567275, 0.43883299236826334, 0.07418012571669322, -0.23361352544134362], [-0.6023789272455896, 0.8341847591891457, -0.08281592798870495, 0.44091093497892936, 0.03406079369180253, -0.230121258467709], [-0.580325096584921, 0.8348600206987563, -0.09423273103613158, 0.4398290658607346, -0.007268916293541752, -0.22598299602198832], [-0.5584055329514385, 0.8334389340511817, -0.10542532203594698, 0.4355384645808257, -0.0496902092050965

...

, -0.05984407237136824, -0.23352686839645492, 0.5012418303177613, 0.0875384140743743], [-0.3213125536134536, -0.7696523042883766, -0.05534307499128104, -0.17142415835618552, 0.5473629195091131, 0.0923188093572605], [-0.3281931463897289, -0.7411480239274214, -0.050603151770641654, -0.10233338246233493, 0.5913185812772008, 0.09708700563009462], [-0.3314335861771757, -0.7104942566048223, -0.04562478151731704, -0.02589194332412617, 0.6333023810455358, 0.10184748273080994], [-0.33065379926277966, -0.6777810236913836, -0.040408358796931707, 0.05841670609425577, 0.6736991850392072, 0.10659862606811647], [-0.3254427649700689, -0.6430724270106188, -0.03495459233881903, 0.15132499986931305, 0.7131796500980409, 0.11132857977825202], [-0.31534557797977536, -0.6063887731640112, -0.029265190689764464, 0.2538726309774979, 0.7528566802719361, 0.11600750744905272], [-0.29984485553164153, -0.5676782860873761, -0.023344095405497898, 0.3675446693804961, 0.7945571597605401, 0.12057258068799483], [-0.278333

, -0.10765348282141876], [0.5639117497956556, 0.17011158429025003, -0.08196233547101918, 0.8933145155772058, 0.3192359586280959, -0.0871613174451067], [0.6070715925712806, 0.18319266259478295, -0.08588139514137962, 0.8327074817543639, 0.20537501758257828, -0.06988939498581088], [0.6473678474474555, 0.19084471900501396, -0.0889959370471163, 0.7782751833583372, 0.10202341480940624, -0.05488496432496784], [0.6850310352435478, 0.19355689849001653, -0.09140234481979018, 0.7271164554634053, 0.007774768051895793, -0.04150197841073719], [0.7201766745327073, 0.19175687445598036, -0.09316936918612534, 0.6774422544108645, -0.07844021161085046, -0.02926558159521601], [0.7528491909283977, 0.18582501667458187, -0.09434453305330126, 0.6281494470182467, -0.15745800360831538, -0.017791545340263913], [0.7830498174073959, 0.17610387362858565, -0.09495717526786261, 0.578586666450436, -0.22996602224554766, -0.006728695045277537], [0.8107556561996992, 0.16290384825437182, -0.09501877892073285, 0.52842596255

 0.17983356402233788, -0.22980609713334266, -0.6732311785833479, -0.029575926375232885, 0.12933078860009636], [-0.7116572288271892, 0.17966948253634893, -0.22262984610021375, -0.6254397182845659, 0.022630257563759006, 0.15677524119629974], [-0.7417960837847091, 0.18205604137626225, -0.21417350060075352, -0.5791097781573399, 0.07227525736331508, 0.180581864091539], [-0.7696427770432587, 0.1868553197133107, -0.20460532926125638, -0.5337790724043165, 0.11899127155381937, 0.20128351448968562], [-0.7952384060135792, 0.19391380960535917, -0.19406954429756956, -0.4891100879424994, 0.162516714226118, 0.21931258733576142], [-0.8186098002655706, 0.20306687338617127, -0.18269078227414592, -0.4448666717115582, 0.2026641337504862, 0.2350219904650926], [-0.8397740988533912, 0.21414188724139907, -0.17057765876195066, -0.40089486431448107, 0.23929899988590989, 0.2487013922994157], [-0.8587424612837237, 0.2269605130700198, -0.1578256050856045, -0.3571073679835157, 0.2723255976239812, 0.2605898054945448

16.216216216216218, 16.266266266266268, 16.316316316316318, 16.366366366366368, 16.416416416416418, 16.466466466466468, 16.516516516516518, 16.566566566566568, 16.616616616616618, 16.666666666666668, 16.716716716716718, 16.766766766766768, 16.816816816816818, 16.866866866866868, 16.916916916916918, 16.966966966966968, 17.017017017017018, 17.067067067067068, 17.117117117117118, 17.167167167167168, 17.217217217217218, 17.26726726726727, 17.31731731731732, 17.36736736736737, 17.41741741741742, 17.46746746746747, 17.51751751751752, 17.56756756756757, 17.61761761761762, 17.66766766766767, 17.71771771771772, 17.76776776776777, 17.81781781781782, 17.86786786786787, 17.91791791791792, 17.96796796796797, 18.01801801801802, 18.06806806806807, 18.11811811811812, 18.16816816816817, 18.21821821821822, 18.26826826826827, 18.31831831831832, 18.36836836836837, 18.41841841841842, 18.46846846846847, 18.51851851851852, 18.56856856856857, 18.61861861861862, 18.66866866866867, 18.71871871871872, 18.7687687

 41.94194194194194, 41.991991991991995, 42.04204204204204, 42.092092092092095, 42.14214214214214, 42.192192192192195, 42.24224224224224, 42.292292292292295, 42.34234234234234, 42.392392392392395, 42.44244244244244, 42.492492492492495, 42.54254254254254, 42.592592592592595, 42.64264264264264, 42.692692692692695, 42.74274274274274, 42.792792792792795, 42.84284284284284, 42.892892892892895, 42.94294294294294, 42.992992992992995, 43.04304304304304, 43.093093093093096, 43.14314314314314, 43.193193193193196, 43.24324324324324, 43.293293293293296, 43.34334334334334, 43.393393393393396, 43.44344344344344, 43.493493493493496, 43.54354354354354, 43.593593593593596, 43.64364364364364, 43.693693693693696, 43.74374374374374, 43.793793793793796, 43.84384384384384, 43.893893893893896, 43.94394394394394, 43.993993993993996, 44.04404404404404, 44.094094094094096, 44.14414414414414, 44.194194194194196, 44.24424424424424, 44.294294294294296, 44.34434434434434, 44.394394394394396, 44.44444444444444, 44.49